In [6]:
import pandas as pd
import numpy as np
import re

In [7]:
gifi = pd.HDFStore('bdv-gifi-all.h5')
size_tuples = []
for gifi_column_key in gifi.keys():
    column_df = gifi.get(gifi_column_key)
    size_tuples.append((gifi_column_key, column_df.size))
    

top_columns = sorted(size_tuples, key=lambda col_values: -1 * col_values[1])
top_columns

[('/total_assets_less_cur._liab._th_gbp', 675367),
 ('/net_assets_th_gbp', 672467),
 ('/net_tangible_assets_(liab.)_th_gbp', 671298),
 ('/total_assets_th_gbp', 666901),
 ('/net_current_assets_(working_capital)_th_gbp', 662191),
 ('/shareholders_funds_th_gbp', 657085),
 ('/current_assets_th_gbp', 646126),
 ('/total_reserves_th_gbp', 605651),
 ('/current_liabilities_th_gbp', 569495),
 ('/solvency_ratio_(asset_based)_(%)', 561279),
 ('/current_ratio_(x)', 548680),
 ('/liquidity_ratio_(x)', 547445),
 ('/total_other_current_liabilities_th_gbp', 543473),
 ('/other_current_assets_th_gbp', 457799),
 ('/bank_&_deposits_th_gbp', 455054),
 ('/other_current_liabilities_th_gbp', 448984),
 ('/profit_(loss)_account_th_gbp', 441665),
 ('/issued_capital_th_gbp', 425106),
 ('/ordinary_shares_th_gbp', 424124),
 ('/other_debtors_th_gbp', 409895),
 ('/fixed_assets_th_gbp', 379093),
 ('/turnover_th_gbp', 353901),
 ('/tangible_assets_th_gbp', 328038),
 ('/ebitda_th_gbp', 293455),
 ('/operating_profit_th_gbp'

In [22]:
def merge_multi(self, df, on):
    return self.reset_index().join(df,on=on).set_index(self.index.names)

pd.DataFrame.merge_multi = merge_multi

column_df = None
joined_df = None
i=0
for (gifi_column_key, size) in top_columns:
    column_df = gifi.get(gifi_column_key)
    # column_df.info(memory_usage=True)
    if joined_df is None:
        joined_df = column_df
        joined_df.info(memory_usage=True)
        
    else:
        #Index.join(self, other[, how, level, …])
        #Index.intersection(self, other[, sort])
        #Index.union(self, other[, sort])
        inddiff = joined_df.index.difference(column_df.index)
        #column_df.info(memory_usage=True)
        print(inddiff.shape)
        #df.set_index('key').join(other.set_index('key'))
        joined_df = joined_df.join(column_df, on=[['bv_id','year']], how='outer', sort=True)
        joined_df.info(memory_usage=True)
    print(joined_df.index.shape)
    #print(joined_df.shape)
    #shape
    i=i+1
    print(str(i))
    if i > 15:
        break
gifi_joined = pd.HDFStore('bdv-gifi-16.h5')
gifi_joined.put('gifi16', joined_df)
gifi_joined.close()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 675367 entries, ('GB07114772', 2011) to ('IE610925', 2019)
Data columns (total 1 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   total_assets_less_cur._liab._th_gbp  675367 non-null  float64
dtypes: float64(1)
memory usage: 14.3+ MB
(675367,)
1


KeyboardInterrupt: 

In [14]:
joined_df.to_csv('gifi16.csv')